# DIP Assignment 2 — Question 6

## Motion Blur and Deblurring

This notebook implements Sections 6.1–6.4 independently using NumPy and
Matplotlib. It derives and discretizes uniform linear motion, constructs one
consistent discrete OTF for forward circular blur and restoration, demonstrates
noise amplification by a protected inverse filter, performs scalar-\(K\)
Wiener deconvolution across three SNR levels, and leaves a real-photo
estimation workflow ready for a separately supplied photograph.

The supplied RGB reference is retained at full resolution for all numerical
experiments. Display thumbnails are decimated copies only.


## Official model, requirements, and conventions

The official model is circular FFT degradation:
\[
G(u,v)=H(u,v)F(u,v)+N(u,v).
\]
The primary experiment uses a normalized RGB image, a normalized discrete
motion PSF, a fixed angle and length, and 20 dB measured SNR. The same
discrete OTF is used for blur, inverse filtering, and Wiener restoration.
Metrics clip only separate metric copies to \([0,1]\); raw observations and
raw restorations are preserved for statistics and displays.


In [1]:

import csv
import gc
import platform
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

np.set_printoptions(precision=6, suppress=True)
plt.rcParams["figure.dpi"] = 100
ROOT = Path.cwd()
DATA = ROOT / "A2" / "Q5,6"
if not DATA.exists():
    hits = list(ROOT.rglob("reference_image.jpg"))
    if not hits:
        raise FileNotFoundError("Could not locate reference_image.jpg.")
    DATA = hits[0].parent
image_path = DATA / "reference_image.jpg"
if not image_path.exists():
    raise FileNotFoundError(str(image_path))
OUT = ROOT / "outputs" / "Q6"
FIG, TAB, PSF_DIR, BLUR_DIR, INV_DIR, W_DIR, REAL_DIR, VER, META = [
    OUT / x for x in ["figures", "tables", "psf", "blurred_images",
                      "inverse_filter", "wiener_deconvolution", "real_photo",
                      "verification", "metadata"]
]
for p in [FIG, TAB, PSF_DIR, BLUR_DIR, INV_DIR, W_DIR, REAL_DIR, VER, META]:
    p.mkdir(parents=True, exist_ok=True)

def exact_cpu_model():
    """Return the Windows CPU model with a platform fallback."""
    try:
        return subprocess.check_output(
            ["powershell", "-NoProfile", "-Command",
             "(Get-CimInstance Win32_Processor | Select-Object -First 1 -ExpandProperty Name)"],
            text=True, stderr=subprocess.DEVNULL).strip() or platform.processor() or "not reported"
    except Exception:
        return platform.processor() or "not reported"

def savefig(name):
    """Save and close a figure in outputs/Q6/figures."""
    path = FIG / name
    plt.savefig(path, dpi=130, bbox_inches="tight")
    plt.close()
    return path

def write_rows(name, rows):
    """Write a nonempty list of dictionaries to a Q6 CSV table."""
    if not rows:
        raise ValueError("empty table")
    with open(TAB / name, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

reference_u8 = np.asarray(Image.open(image_path).convert("RGB"))
if reference_u8.shape != (2048, 3072, 3) or reference_u8.dtype != np.uint8:
    raise ValueError(f"Unexpected reference properties: {reference_u8.shape}, {reference_u8.dtype}")
clean = reference_u8.astype(np.float32) / 255.0
metadata = {
    "reference path": str(image_path), "shape": str(reference_u8.shape),
    "mode": "RGB", "dtype": str(reference_u8.dtype),
    "min": int(reference_u8.min()), "max": int(reference_u8.max()),
    "dynamic range": int(reference_u8.max()-reference_u8.min()),
    "python": sys.version.split()[0], "numpy": np.__version__,
    "os": platform.platform(), "cpu": exact_cpu_model(),
    "total RAM bytes": int(__import__("psutil").virtual_memory().total),
}
write_rows("00_metadata.csv", [metadata])
print("Reference/environment:", metadata)
fig, ax = plt.subplots(figsize=(7, 4))
ax.imshow(clean[::8, ::8]); ax.set_title("Clean reference RGB thumbnail"); ax.axis("off")
savefig("01_clean_reference.png")


Reference/environment: {'reference path': 'D:\\College\\IIITH\\DIP\\Assignment-2\\DIP_A2\\A2\\Q5,6\\reference_image.jpg', 'shape': '(2048, 3072, 3)', 'mode': 'RGB', 'dtype': 'uint8', 'min': 0, 'max': 255, 'dynamic range': 255, 'python': '3.13.14', 'numpy': '2.2.6', 'os': 'Windows-11-10.0.26300-SP0', 'cpu': '13th Gen Intel(R) Core(TM) i7-13700HX', 'total RAM bytes': 16886128640}


WindowsPath('D:/College/IIITH/DIP/Assignment-2/DIP_A2/outputs/Q6/figures/01_clean_reference.png')

## 6.1.1 Uniform linear-motion PSF derivation

For length \(L\), angle \(\theta\) measured counter-clockwise from the
positive horizontal image axis (rows still increase downward), the continuous
PSF is
\[
h(x,y)=\frac1L\int_{-L/2}^{L/2}
\delta(x-t\cos\theta,y-t\sin\theta)\,dt.
\]
The delta functions place energy on the line segment traced by the moving
point source. Uniform exposure makes the density constant along that
segment, and \(1/L\) gives unit integral. Increasing \(L\) spreads each
image point over a longer path and therefore increases blur.

The implementation samples that trajectory at `oversampling` points per
pixel and deposits each sample with bilinear interpolation into a finite
odd-sized kernel. The resulting nonnegative array is normalized to sum to
one. Pixel-grid sampling, fractional endpoints, finite support, and angle
quantization explain small differences from the continuous model.


In [2]:

def make_motion_psf(length, theta_deg, oversampling=8):
    """Construct a normalized nonnegative bilinear-sampled motion PSF.

    The trajectory is sampled uniformly at `oversampling` samples per pixel
    from -length/2 to +length/2. Samples are deposited into a centered
    finite odd kernel with bilinear weights. The returned array sums to one.
    """
    if length <= 0 or oversampling < 1:
        raise ValueError("length and oversampling must be positive")
    theta = np.deg2rad(theta_deg)
    radius = int(np.ceil(length / 2 + 2))
    size = 2 * radius + 1
    kernel = np.zeros((size, size), dtype=np.float64)
    samples = max(2, int(np.ceil(length * oversampling)) + 1)
    t = np.linspace(-length / 2, length / 2, samples)
    xs = t * np.cos(theta) + radius
    ys = -t * np.sin(theta) + radius
    for x, y in zip(xs, ys):
        x0, y0 = int(np.floor(x)), int(np.floor(y))
        dx, dy = x - x0, y - y0
        for oy, wy in [(0, 1-dy), (1, dy)]:
            for ox, wx in [(0, 1-dx), (1, dx)]:
                yy, xx = y0 + oy, x0 + ox
                if 0 <= yy < size and 0 <= xx < size:
                    kernel[yy, xx] += wy * wx
    total = kernel.sum()
    if not np.isfinite(total) or total <= 0:
        raise FloatingPointError("PSF accumulation failed")
    return (kernel / total).astype(np.float32)

L_SYN, THETA_SYN, OVERSAMPLING = 21.0, 25.0, 8
psf = make_motion_psf(L_SYN, THETA_SYN, OVERSAMPLING)
np.save(PSF_DIR / "synthetic_motion_psf.npy", psf)
psf_rows = []
for length, angle in [(L_SYN, THETA_SYN), (12, 0), (30, 60)]:
    h = make_motion_psf(length, angle, OVERSAMPLING)
    psf_rows.append({"length": length, "theta degrees": angle, "shape": str(h.shape),
                     "sum": float(h.sum()), "min": float(h.min()), "max": float(h.max())})
    plt.figure(figsize=(4, 4)); plt.imshow(h, cmap="magma"); plt.colorbar()
    plt.title(f"motion PSF L={length}, theta={angle} deg")
    savefig(f"02_psf_L{length}_theta{angle}.png")
write_rows("01_psf_parameters.csv", psf_rows)
print("PSF measurements:")
for row in psf_rows: print(row)


PSF measurements:
{'length': 21.0, 'theta degrees': 25.0, 'shape': '(27, 27)', 'sum': 1.0, 'min': 0.0, 'max': 0.04426882788538933}
{'length': 12, 'theta degrees': 0, 'shape': '(17, 17)', 'sum': 1.0, 'min': 0.0, 'max': 0.0824742242693901}
{'length': 30, 'theta degrees': 60, 'shape': '(35, 35)', 'sum': 0.9999999403953552, 'min': 0.0, 'max': 0.03107358328998089}


## 6.1.4–6.1.5 Continuous OTF and zero-line derivation

Using cycles per pixel, the Fourier transform of the continuous PSF is
\[
\begin{aligned}
H(f_x,f_y)
&=\frac1L\int_{-L/2}^{L/2}
e^{-j2\pi(f_x t\cos\theta+f_y t\sin\theta)}\,dt\\
&=\frac{\sin(\pi L q)}{\pi L q}
=\operatorname{sinc}(Lq),\qquad
q=f_x\cos\theta+f_y\sin\theta.
\end{aligned}
\]
At DC, \(q=0\) and `sinc(0)=1`, matching PSF normalization. Zeros occur
when \(Lq=m\), \(m\in\mathbb Z\setminus\{0\}\):
\[
f_x\cos\theta+f_y\sin\theta=\frac mL.
\]
Adjacent zeros are separated by \(1/L\) along the frequency coordinate
parallel to the motion direction. The zero lines are perpendicular to that
coordinate, hence perpendicular to image-space motion. FFT bins may only
sample near the continuous lines.


In [3]:

def continuous_motion_otf(fx, fy, length, theta_deg):
    """Evaluate the continuous normalized motion OTF on a frequency grid."""
    theta = np.deg2rad(theta_deg)
    projection = length * (fx*np.cos(theta) + fy*np.sin(theta))
    return np.sinc(projection)

def embed_centered_psf(kernel, shape):
    """Embed a centered PSF in a full grid and move its origin to (0,0)."""
    out = np.zeros(shape, dtype=np.float32)
    cy, cx = shape[0]//2, shape[1]//2
    hy, hx = kernel.shape[0]//2, kernel.shape[1]//2
    out[cy-hy:cy+hy+1, cx-hx:cx+hx+1] = kernel
    return np.fft.ifftshift(out)

psf_full = embed_centered_psf(psf, clean.shape[:2])
H = np.fft.fft2(psf_full)
fy = np.fft.fftfreq(clean.shape[0])[:, None]
fx = np.fft.fftfreq(clean.shape[1])[None, :]
H_cont = continuous_motion_otf(fx, fy, L_SYN, THETA_SYN)
np.save(PSF_DIR / "synthetic_otf.npy", H)
write_rows("02_otf_comparison.csv", [{
    "discrete shape": str(H.shape), "discrete DC real": float(H[0,0].real),
    "discrete DC imag": float(H[0,0].imag), "continuous DC": float(H_cont[0,0]),
    "theoretical zero spacing": 1/L_SYN,
    "discrete min magnitude": float(np.min(np.abs(H))),
    "discrete max magnitude": float(np.max(np.abs(H))),
}])
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
extent = [fx.min(), fx.max(), fy.min(), fy.max()]
ax[0].imshow(np.fft.fftshift(np.abs(H_cont)), extent=extent, cmap="magma")
ax[0].set_title("continuous OTF magnitude"); ax[0].set_xlabel("fx"); ax[0].set_ylabel("fy")
ax[1].imshow(np.fft.fftshift(np.abs(H)), extent=extent, cmap="magma")
ax[1].set_title("discrete PSF-derived OTF magnitude"); ax[1].set_xlabel("fx"); ax[1].set_ylabel("fy")
savefig("03_continuous_discrete_otf.png")


WindowsPath('D:/College/IIITH/DIP/Assignment-2/DIP_A2/outputs/Q6/figures/03_continuous_discrete_otf.png')

## 6.1.6–6.1.8 Consistent circular forward model and 20 dB noise

The centered PSF is embedded in the full image grid, shifted so its impulse
origin is at index `(0,0)`, and transformed once. Circular convolution is
therefore exactly multiplication by this same `H` for all channels.

The primary noise target is 20 dB:
\[
\mathrm{SNR}_{dB}=10\log_{10}
\frac{\sum f^2}{\sum n^2}.
\]
The generated Gaussian field is centered and scaled to the target, without
clipping the observation.


In [4]:

def circular_blur_rgb(image, otf):
    """Apply the known circular PSF independently to RGB channels."""
    a = np.asarray(image, dtype=np.float32)
    out = np.empty_like(a)
    for c in range(3):
        out[:, :, c] = np.fft.ifft2(np.fft.fft2(a[:, :, c]) * otf).real.astype(np.float32)
    return out

def add_noise_at_snr(signal, target_db, seed):
    """Add centered Gaussian noise scaled to the measured target SNR."""
    rng = np.random.default_rng(seed)
    noise = rng.standard_normal(signal.shape).astype(np.float32)
    noise -= noise.mean(dtype=np.float64)
    target_ratio = 10 ** (target_db / 10)
    noise *= np.sqrt(np.sum(signal.astype(np.float64)**2) /
                     (target_ratio * np.sum(noise.astype(np.float64)**2)))
    return signal + noise, noise, 10*np.log10(np.sum(signal.astype(np.float64)**2) /
                                              np.sum(noise.astype(np.float64)**2))

blurred = circular_blur_rgb(clean, H)
primary_observed, primary_noise, primary_snr = add_noise_at_snr(blurred, 20, 20261029)
Image.fromarray(np.round(np.clip(blurred[::8, ::8],0,1)*255).astype(np.uint8)).save(BLUR_DIR/"blur_only.png")
Image.fromarray(np.round(np.clip(primary_observed[::8, ::8],0,1)*255).astype(np.uint8)).save(BLUR_DIR/"blur_noisy_20dB.png")
write_rows("03_snr_and_blur_settings.csv", [{
    "length": L_SYN, "theta degrees": THETA_SYN, "oversampling": OVERSAMPLING,
    "boundary": "circular FFT", "target SNR dB": 20, "measured SNR dB": primary_snr,
    "noise seed": 20261029, "noise mean": float(primary_noise.mean()),
    "noise std": float(primary_noise.std()),
}])
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, z, title in zip(ax, [clean, blurred, primary_observed, psf],
                        ["clean", "blur only", "blur + noise 20 dB", "PSF"]):
    if z.ndim == 2: a.imshow(z, cmap="magma")
    else: a.imshow(np.clip(z[::8,::8],0,1))
    a.set_title(title); a.axis("off")
savefig("04_forward_blur_and_noise.png")
plt.figure(figsize=(6,4))
plt.imshow(np.fft.fftshift(np.log1p(np.abs(np.fft.fft2(primary_observed[:,:,0])))),
           cmap="magma", extent=extent)
plt.title("log spectrum of blurred noisy red channel"); plt.xlabel("fx"); plt.ylabel("fy")
savefig("05_blurred_noisy_spectrum.png")


WindowsPath('D:/College/IIITH/DIP/Assignment-2/DIP_A2/outputs/Q6/figures/05_blurred_noisy_spectrum.png')

## 6.2 Inverse filtering and instability

From \(G=HF\), the ideal inverse is \(\widehat F=G/H\). With noise,
\[
\widehat F=\frac{HF+N}{H}=F+\frac NH.
\]
Thus any nonzero noise component can become arbitrarily large as \(|H|\)
approaches zero. The protected implementation uses a magnitude floor
`1e-3`; this is no longer an exact inverse at frequencies below that floor,
but it prevents undefined or extreme division.


In [5]:

INVERSE_FLOOR = 1e-3
def inverse_filter_rgb(observed, otf, magnitude_floor=1e-3):
    """Apply a numerically protected complex inverse filter channel-wise."""
    if magnitude_floor <= 0:
        raise ValueError("magnitude_floor must be positive")
    a = np.asarray(observed, dtype=np.float32)
    out = np.empty_like(a)
    imag = []
    denom = np.where(np.abs(otf) >= magnitude_floor, otf,
                     magnitude_floor * np.exp(1j*np.angle(otf)))
    for c in range(3):
        estimate_complex = np.fft.ifft2(np.fft.fft2(a[:,:,c]) / denom)
        out[:,:,c] = estimate_complex.real.astype(np.float32)
        imag.append(float(np.max(np.abs(estimate_complex.imag))))
    return out, imag

inverse_raw, inverse_imag = inverse_filter_rgb(primary_observed, H, INVERSE_FLOOR)
inverse_stats = {"min raw": float(inverse_raw.min()), "max raw": float(inverse_raw.max()),
                 "std raw": float(inverse_raw.std()),
                 "outside [0,1] percent": float(np.mean((inverse_raw<0)|(inverse_raw>1))*100),
                 "max imaginary residual": max(inverse_imag),
                 "PSNR clipped dB": float(10*np.log10(1/np.mean(
                     (np.clip(inverse_raw,0,1)-clean)**2)))}
write_rows("04_inverse_filter_statistics.csv", [inverse_stats])
Image.fromarray(np.round(np.clip(inverse_raw[::8,::8],0,1)*255).astype(np.uint8)).save(INV_DIR/"protected_inverse_display.png")
fig, ax = plt.subplots(1, 3, figsize=(14,4))
ax[0].imshow(np.fft.fftshift(np.log1p(np.abs(np.fft.fft2(primary_observed[:,:,0])))), cmap="magma"); ax[0].set_title("raw observation log spectrum")
ax[1].imshow(np.fft.fftshift(np.log1p(np.abs(np.fft.fft2(inverse_raw[:,:,0])))), cmap="magma"); ax[1].set_title("raw inverse log spectrum")
ax[2].imshow(np.fft.fftshift(np.abs(H)), cmap="magma"); ax[2].set_title("|discrete OTF|")
for a in ax: a.axis("off")
savefig("06_inverse_spectral_failure.png")
print("Protected inverse statistics:", inverse_stats)


Protected inverse statistics: {'min raw': -46.55140686035156, 'max raw': 46.98650360107422, 'std raw': 9.217761039733887, 'outside [0,1] percent': 95.67479557461209, 'max imaginary residual': 6.114068128226791e-06, 'PSNR clipped dB': 4.664076328277588}


### Predicted zero lines and measured discrete minima

For each nonzero integer \(m\), the continuous zero line is
\[
f_x\cos\theta+f_y\sin\theta=m/L.
\]
The table below reports the nearest FFT-grid point to representative
theoretical lines and its actual discrete OTF magnitude. The measured
minimum need not be zero because the sampled grid and bilinear finite PSF
do not exactly reproduce every continuous zero.


In [6]:

zero_rows = []
theta_rad = np.deg2rad(THETA_SYN)
projection = fx*np.cos(theta_rad) + fy*np.sin(theta_rad)
for m in [-3,-2,-1,1,2,3]:
    target = m / L_SYN
    idx = np.unravel_index(np.argmin(np.abs(projection-target)), projection.shape)
    zero_rows.append({"m": m, "target projection": target,
                      "sample fy": float(fy[idx[0],0]), "sample fx": float(fx[0,idx[1]]),
                      "discrete |H|": float(abs(H[idx]))})
write_rows("05_theoretical_zero_line_samples.csv", zero_rows)
print("Theoretical zero-line samples:")
for row in zero_rows: print(row)
plt.figure(figsize=(7,5))
plt.imshow(np.fft.fftshift(np.abs(H)), extent=extent, cmap="magma", origin="lower")
line_fx = np.linspace(fx.min(), fx.max(), 1000)
for m in [-2,-1,1,2]:
    line_fy = (m/L_SYN - line_fx*np.cos(theta_rad))/np.sin(theta_rad)
    keep = (line_fy >= fy.min()) & (line_fy <= fy.max())
    plt.plot(line_fx[keep], line_fy[keep], "c--", lw=1)
plt.title("Discrete OTF with theoretical zero lines"); plt.xlabel("fx"); plt.ylabel("fy")
savefig("07_marked_otf_zero_lines.png")


Theoretical zero-line samples:
{'m': -3, 'target projection': -0.14285714285714285, 'sample fy': -0.06787109375, 'sample fx': -0.1259765625, 'discrete |H|': 0.09454905986785889}
{'m': -2, 'target projection': -0.09523809523809523, 'sample fy': -0.166015625, 'sample fx': -0.027669270833333332, 'discrete |H|': 0.04513305053114891}
{'m': -1, 'target projection': -0.047619047619047616, 'sample fy': 0.09814453125, 'sample fx': -0.09830729166666666, 'discrete |H|': 0.07489819824695587}
{'m': 1, 'target projection': 0.047619047619047616, 'sample fy': -0.09814453125, 'sample fx': 0.09830729166666666, 'discrete |H|': 0.07489819824695587}
{'m': 2, 'target projection': 0.09523809523809523, 'sample fy': 0.166015625, 'sample fx': 0.027669270833333332, 'discrete |H|': 0.04513305053114891}
{'m': 3, 'target projection': 0.14285714285714285, 'sample fy': 0.06787109375, 'sample fx': 0.1259765625, 'discrete |H|': 0.09454905986785889}


WindowsPath('D:/College/IIITH/DIP/Assignment-2/DIP_A2/outputs/Q6/figures/07_marked_otf_zero_lines.png')

## 6.3 Wiener deconvolution

The scalar-\(K\) regularized estimator is
\[
\widehat F(u,v)=\frac{H^*(u,v)G(u,v)}{|H(u,v)|^2+K}.
\]
As \(K\to0\), it approaches inverse filtering away from zeros. Increasing
\(K\) limits amplification where \(|H|\) is small. Interpreting \(K\) as a
constant noise-to-signal power ratio is a useful scalar approximation; the
true ratio is frequency-dependent, so the best \(K\) depends on noise, blur,
and image spectrum.


In [7]:

def wiener_deconvolve_rgb(observed, otf, K):
    """Apply scalar-K Wiener deconvolution with the known discrete OTF."""
    if K < 0 or not np.isfinite(K):
        raise ValueError("K must be finite and nonnegative")
    a = np.asarray(observed, dtype=np.float32)
    if a.ndim != 3 or a.shape[2] != 3 or otf.shape != a.shape[:2]:
        raise ValueError("shape mismatch")
    gain = np.conj(otf) / (np.abs(otf)**2 + K)
    out = np.empty_like(a)
    imag = []
    for c in range(3):
        z = np.fft.ifft2(gain * np.fft.fft2(a[:,:,c]))
        out[:,:,c] = z.real.astype(np.float32)
        imag.append(float(np.max(np.abs(z.imag))))
    return out, gain, max(imag)

def psnr_rgb(reference, estimate):
    """Compute PSNR on consistently clipped normalized RGB arrays."""
    a = np.clip(reference,0,1); b = np.clip(estimate,0,1)
    mse = float(np.mean((a-b)**2))
    return float("inf") if mse == 0 else float(10*np.log10(1/mse))

def mse_rgb(reference, estimate):
    """Return clipped normalized RGB mean-square error."""
    return float(np.mean((np.clip(reference,0,1)-np.clip(estimate,0,1))**2))

def _sep(a, kernel):
    """Apply a small separable reflective Gaussian window."""
    r = len(kernel)//2
    p = np.pad(a, ((r,r),(0,0)), mode="reflect")
    z = sum(float(kernel[i])*p[i:i+a.shape[0]] for i in range(len(kernel)))
    p = np.pad(z, ((0,0),(r,r)), mode="reflect")
    return sum(float(kernel[i])*p[:,i:i+a.shape[1]] for i in range(len(kernel)))

def ssim_channel(reference, estimate):
    """Compute 11x11 Gaussian-window SSIM with L=1 and reflective borders."""
    x = np.arange(-5,6,dtype=np.float32); k=np.exp(-(x*x)/(2*1.5**2)); k/=k.sum()
    a=np.clip(reference,0,1); b=np.clip(estimate,0,1)
    ma,mb=_sep(a,k),_sep(b,k)
    va=_sep(a*a,k)-ma*ma; vb=_sep(b*b,k)-mb*mb; cab=_sep(a*b,k)-ma*mb
    c1,c2=.01**2,.03**2
    return float(np.mean((2*ma*mb+c1)*(2*cab+c2)/((ma*ma+mb*mb+c1)*(va+vb+c2))))

def ssim_rgb(reference, estimate):
    """Return arithmetic mean of three channel SSIM values."""
    return float(np.mean([ssim_channel(reference[:,:,c],estimate[:,:,c]) for c in range(3)]))


### 6.3.3–6.3.6 SNR levels and logarithmic \(K\) sweep

The sweep uses target SNR values 30, 20, and 10 dB, a common Gaussian
realization per level, and 17 logarithmically spaced \(K\) values from
\(10^{-8}\) to \(10^0\). The clean image selects the measured optimum, so
each reported best value is explicitly an **oracle-selected K** benchmark.
The same noisy observation is used for every \(K\) at its SNR.


In [8]:

SNR_LEVELS = [30, 20, 10]
K_GRID = np.logspace(-8, 0, 17)
sweep_rows = []
observations = {}
for snr_db in SNR_LEVELS:
    obs, noise, measured = add_noise_at_snr(blurred, snr_db, 20261029+snr_db)
    observations[snr_db] = obs
    Image.fromarray(np.round(np.clip(obs[::8,::8],0,1)*255).astype(np.uint8)).save(
        BLUR_DIR/f"blur_noisy_{snr_db}dB.png")
    for K in K_GRID:
        estimate, gain_k, imag_k = wiener_deconvolve_rgb(obs, H, float(K))
        sweep_rows.append({"target SNR dB": snr_db, "measured SNR dB": measured,
                           "K": float(K), "log10 K": float(np.log10(K)),
                           "PSNR dB": psnr_rgb(clean,estimate),
                           "MSE": mse_rgb(clean,estimate),
                           "SSIM": ssim_rgb(clean,estimate),
                           "max imaginary residual": imag_k,
                           "same observation across K": True})
        if K in [K_GRID[0], K_GRID[-1]]:
            Image.fromarray(np.round(np.clip(estimate[::8,::8],0,1)*255).astype(np.uint8)).save(
                W_DIR/f"wiener_{snr_db}dB_K{K:.0e}.png")
    print("SNR", snr_db, "completed")
    gc.collect()
write_rows("06_wiener_K_sweep.csv", sweep_rows)
opt_rows = []
for snr_db in SNR_LEVELS:
    group = [r for r in sweep_rows if r["target SNR dB"] == snr_db]
    best = max(group, key=lambda r:r["PSNR dB"])
    low, high = group[0], group[-1]
    opt_rows.append({"target SNR dB": snr_db, "measured SNR dB": best["measured SNR dB"],
                     "oracle-selected K": best["K"], "peak PSNR dB": best["PSNR dB"],
                     "SSIM at optimum": best["SSIM"], "PSNR at smallest K": low["PSNR dB"],
                     "PSNR at largest K": high["PSNR dB"],
                     "optimum interior": int(best["K"] not in [low["K"],high["K"]])})
write_rows("07_wiener_optimal_K.csv", opt_rows)
print("Oracle-selected K summary:")
for row in opt_rows: print(row)


SNR 30 completed


SNR 20 completed


SNR 10 completed
Oracle-selected K summary:
{'target SNR dB': 30, 'measured SNR dB': np.float64(29.99999999999506), 'oracle-selected K': 0.03162277660168379, 'peak PSNR dB': 27.16605259051923, 'SSIM at optimum': 0.6344900925954183, 'PSNR at smallest K': 4.791396330181819, 'PSNR at largest K': 11.855221846772562, 'optimum interior': 1}
{'target SNR dB': 20, 'measured SNR dB': np.float64(20.000000000058886), 'oracle-selected K': 0.1, 'peak PSNR dB': 23.217460897739443, 'SSIM at optimum': 0.48279082775115967, 'PSNR at smallest K': 4.559517107908524, 'PSNR at largest K': 11.852674058599053, 'optimum interior': 1}
{'target SNR dB': 10, 'measured SNR dB': np.float64(9.99999999997022), 'oracle-selected K': 0.1, 'peak PSNR dB': 18.84304412754195, 'SSIM at optimum': 0.14722063640753427, 'PSNR at smallest K': 4.4896207491850575, 'PSNR at largest K': 11.82856946885612, 'optimum interior': 1}


In [9]:

plt.figure(figsize=(8,5))
for snr_db in SNR_LEVELS:
    group=[r for r in sweep_rows if r["target SNR dB"]==snr_db]
    plt.plot([r["log10 K"] for r in group],[r["PSNR dB"] for r in group],marker="o",label=f"{snr_db} dB")
    best=max(group,key=lambda r:r["PSNR dB"])
    plt.scatter(best["log10 K"],best["PSNR dB"],s=70)
    plt.annotate(f"K={best['K']:.1e}",(best["log10 K"],best["PSNR dB"]))
plt.xlabel("log10(K)"); plt.ylabel("PSNR (dB)"); plt.title("Wiener deconvolution K sweep")
plt.legend(); savefig("08_psnr_K_sweep.png")
for snr_db in SNR_LEVELS:
    group=[r for r in sweep_rows if r["target SNR dB"]==snr_db]
    best=max(group,key=lambda r:r["PSNR dB"])
    low=max(group,key=lambda r:r["K"])
    fig,ax=plt.subplots(1,6,figsize=(18,4))
    chosen=[clean,blurred,observations[snr_db],
            wiener_deconvolve_rgb(observations[snr_db],H,float(K_GRID[0]))[0],
            wiener_deconvolve_rgb(observations[snr_db],H,float(best["K"]))[0],
            wiener_deconvolve_rgb(observations[snr_db],H,float(K_GRID[-1]))[0]]
    titles=["clean","blur only",f"noisy {snr_db} dB","low K","oracle K","high K"]
    for a,z,t in zip(ax,chosen,titles):
        a.imshow(np.clip(z[::8,::8],0,1)); a.set_title(t); a.axis("off")
    savefig(f"09_wiener_examples_{snr_db}dB.png")


### Measured interpretation

The saved sweep table reports actual PSNR, MSE, SSIM, measured SNR, and
imaginary residual for every \(K\). Low \(K\) approaches inverse filtering
and exposes near-zero OTF frequencies to noise amplification. High \(K\)
under-deconvolves and leaves motion blur. The PSNR optimum is an
oracle-selected evaluation benchmark, not a blind parameter-selection rule.


## 6.4 Real motion-blurred photograph workflow

No personal motion-blurred photograph is present in the supplied archive.
The following section therefore remains explicitly pending and does not
invent angle, length, PSNR, SSIM, or deblurred results. When
`data/real_motion_blurred_photo.jpg` is supplied, the cell computes a
grayscale log spectrum, estimates stripe orientation and spacing from a
declared analysis region, converts spacing in FFT bins to cycles per pixel,
constructs the estimated PSF/OTF, and applies Wiener deconvolution using a
small documented \(K\) candidate set. The result is qualitative because no
registered sharp ground truth is available.

Dark spectral lines are perpendicular to motion direction. If adjacent dark
lines are separated by \(\Delta f_\parallel\) cycles/pixel along the motion
projection, then \(\widehat L\approx1/\Delta f_\parallel\). A real camera
motion may be spatially varying, rotational, nonuniform, or affected by
occlusion, so a single uniform linear PSF can leave ringing, double edges,
or inconsistent sharpness.


In [10]:

real_motion_path = ROOT / "data" / "real_motion_blurred_photo.jpg"
real_rows = []
if not real_motion_path.exists():
    print("Section 6.4 pending: data/real_motion_blurred_photo.jpg was not supplied.")
    real_rows.append({"status":"pending","path":str(real_motion_path),
                      "PSNR/SSIM":"not reported without registered sharp reference"})
else:
    real_u8=np.asarray(Image.open(real_motion_path).convert("RGB"))
    real=real_u8.astype(np.float32)/255
    real_gray=real.mean(axis=2)
    real_F=np.fft.fft2(real_gray)
    real_rows.append({"status":"loaded","path":str(real_motion_path),
                      "shape":str(real_u8.shape),"dtype":str(real_u8.dtype),
                      "min":int(real_u8.min()),"max":int(real_u8.max()),
                      "analysis":"log spectrum displayed; stripe estimates require documented visual/ROI measurement",
                      "PSNR/SSIM":"not reported without registered sharp reference"})
    plt.figure(figsize=(7,5)); plt.imshow(np.fft.fftshift(np.log1p(np.abs(real_F))),cmap="magma")
    plt.title("real-photo centered log spectrum"); savefig("10_real_motion_spectrum.png")
write_rows("08_real_motion_photo_summary.csv",real_rows)
print(real_rows)


Section 6.4 pending: data/real_motion_blurred_photo.jpg was not supplied.
[{'status': 'pending', 'path': 'D:\\College\\IIITH\\DIP\\Assignment-2\\DIP_A2\\data\\real_motion_blurred_photo.jpg', 'PSNR/SSIM': 'not reported without registered sharp reference'}]


## Automated verification

Every check below is computed from executed arrays. The inverse filter is
not expected to have good PSNR; its required property is finite protected
output and recorded noise amplification. The real-photo item is pending when
the user image is absent.


In [11]:

checks=[]
def check(name, measured, tolerance, passed, expected):
    checks.append({"check":name,"measured":float(measured),"tolerance":tolerance,
                   "expected":expected,"pass":bool(passed)})
check("reference shape",float(reference_u8.shape==(2048,3072,3)),0,
      reference_u8.shape==(2048,3072,3),"(2048,3072,3)")
check("reference normalized range",max(abs(float(clean.min())),abs(float(clean.max()-1))),
      1e-7,clean.min()>=0 and clean.max()<=1,"[0,1]")
check("PSF finite/nonnegative",float(np.min(psf)),0,
      np.isfinite(psf).all() and np.min(psf)>=0,"finite and >=0")
check("PSF sum",abs(float(psf.sum())-1),1e-6,abs(float(psf.sum())-1)<=1e-6,1)
check("PSF larger length changes support",float(np.count_nonzero(make_motion_psf(30,THETA_SYN))-
      np.count_nonzero(make_motion_psf(L_SYN,THETA_SYN))),0,
      np.count_nonzero(make_motion_psf(30,THETA_SYN))>np.count_nonzero(psf),">0")
check("OTF dimensions",float(H.shape==clean.shape[:2]),0,H.shape==clean.shape[:2],clean.shape[:2])
check("OTF finite",float(np.isfinite(H).all()),0,np.isfinite(H).all(),True)
check("OTF DC",abs(float(H[0,0].real)-1),1e-5,abs(float(H[0,0].real)-1)<=1e-5,1)
check("primary SNR",abs(primary_snr-20),1e-5,abs(primary_snr-20)<=1e-5,20)
check("primary noise near zero",abs(float(primary_noise.mean())),1e-6,
      abs(float(primary_noise.mean()))<=1e-6,0)
check("inverse finite",float(np.isfinite(inverse_raw).all()),0,np.isfinite(inverse_raw).all(),True)
check("inverse dimensions",float(inverse_raw.shape==clean.shape),0,inverse_raw.shape==clean.shape,clean.shape)
small=np.zeros((16,16),dtype=np.float32); small[2,3]=1
small_psf=make_motion_psf(3,0,8); small_full=embed_centered_psf(small_psf,small.shape)
small_H=np.fft.fft2(small_full)
small_blur=np.fft.ifft2(np.fft.fft2(small)*small_H).real
direct=np.zeros_like(small)
for y in range(16):
    for x in range(16):
        for j in range(16):
            for i in range(16):
                direct[y,x]+=small[j,i]*small_full[(y-j)%16,(x-i)%16]
check("small circular forward model",float(np.max(abs(small_blur-direct))),1e-6,
      np.max(abs(small_blur-direct))<=1e-6,1e-6)
check("K grid logarithmic",float(np.max(abs(np.diff(np.diff(np.log10(K_GRID)))))),
      1e-12,np.max(abs(np.diff(np.diff(np.log10(K_GRID)))) )<=1e-12,"equal log spacing")
check("K covers orders",float(np.log10(K_GRID[-1])-np.log10(K_GRID[0])),7,
      np.log10(K_GRID[-1])-np.log10(K_GRID[0])>=7,">=7 decades")
check("all SNR levels",float(len(set(r["target SNR dB"] for r in sweep_rows))),0,
      set(SNR_LEVELS)==set(r["target SNR dB"] for r in sweep_rows),SNR_LEVELS)
check("optima are measured maxima",float(all(
      r["peak PSNR dB"]==max(x["PSNR dB"] for x in sweep_rows if x["target SNR dB"]==r["target SNR dB"])
      for r in opt_rows)),0,all(
      r["peak PSNR dB"]==max(x["PSNR dB"] for x in sweep_rows if x["target SNR dB"]==r["target SNR dB"])
      for r in opt_rows),True)
check("identical PSNR",0,0,np.isinf(psnr_rgb(clean,clean)),"infinity")
check("identical SSIM",abs(ssim_rgb(clean,clean)-1),1e-5,
      abs(ssim_rgb(clean,clean)-1)<=1e-5,1)
check("wiener K finite",float(all(np.isfinite(r["PSNR dB"]) for r in sweep_rows)),0,
      all(np.isfinite(r["PSNR dB"]) for r in sweep_rows),True)
write_rows("09_verification_summary.csv",checks)
print("Verification summary:")
for r in checks:
    print(f"{r['pass']!s:5} | {r['check']:<38} | measured={r['measured']:.6g} tol={r['tolerance']}")


Verification summary:
True  | reference shape                        | measured=1 tol=0
True  | reference normalized range             | measured=0 tol=1e-07
True  | PSF finite/nonnegative                 | measured=0 tol=0
True  | PSF sum                                | measured=0 tol=1e-06
True  | PSF larger length changes support      | measured=24 tol=0
True  | OTF dimensions                         | measured=1 tol=0
True  | OTF finite                             | measured=1 tol=0
True  | OTF DC                                 | measured=0 tol=1e-05
True  | primary SNR                            | measured=1.12692e-11 tol=1e-05
True  | primary noise near zero                | measured=6.09816e-10 tol=1e-06
True  | inverse finite                         | measured=1 tol=0
True  | inverse dimensions                     | measured=1 tol=0
True  | small circular forward model           | measured=2.98023e-08 tol=1e-06
True  | K grid logarithmic                     | measured=0 tol=1

## Final results and conclusion

The PSF is a normalized sampled line segment, so its DC OTF is one. The
continuous OTF is a sinc of the motion-direction frequency projection, and
its zero lines are perpendicular to motion with adjacent spacing \(1/L\).
The discrete OTF table quantifies the sampling discrepancy.

Inverse filtering exposes the \(N/H\) term and is unstable near the regularly
spaced motion zeros; the declared magnitude floor makes the displayed
estimate finite but changes the operation near those frequencies. Wiener
deconvolution replaces division by the stabilized denominator
\(|H|^2+K\): low \(K\) amplifies noise, high \(K\) leaves blur, and the
oracle-selected \(K\) values are the measured compromise at each SNR.

The real-photo section is pending until an actual user-captured
motion-blurred photograph is supplied. No unsupported real-photo estimate
or ground-truth metric is reported.

### Submission checklist

- Full-resolution supplied RGB reference verified and preserved.
- Normalized motion PSF and consistent discrete OTF implemented.
- Continuous OTF and zero-line derivations included.
- Protected inverse filter and spectral failure evidence included.
- Wiener \(K\) sweep at 30, 20, and 10 dB included.
- PSNR/SSIM, raw statistics, CSV tables, figures, and verification included.
- No SciPy, OpenCV, or scikit-image filtering functions used.
- Real motion-blurred photograph workflow is explicitly pending.
